# Unit 5: Finance — Quantum Amplitude Estimation

**Companion notebook for *What Quantum Computers Are Actually For***

We price a European call option using quantum amplitude estimation on a cloud
[Quokka](https://www.quokkacomputing.com/), and compare convergence with classical
Monte Carlo.

**What you'll see:**
1. Compute the Black-Scholes analytical price (the correct answer)
2. Classical Monte Carlo: watch the $1/\sqrt{N}$ convergence
3. Build a Grover oracle for the payoff function
4. Run amplitude estimation on Quokka
5. Compare convergence rates

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

import requests
from requests.packages.urllib3.exceptions import InsecureRequestWarning
requests.packages.urllib3.disable_warnings(InsecureRequestWarning)

QUOKKA = "quokka1"
QUOKKA_URL = f"http://{QUOKKA}.quokkacomputing.com/qsim/qasm"

def run_qasm(program: str, shots: int = 1024) -> dict:
    result = requests.post(QUOKKA_URL, json={"script": program, "count": shots}, verify=False)
    result.raise_for_status()
    return json.loads(result.content)

print(f"Connected to {QUOKKA_URL}")

## 1. The option and its analytical price

European call option: right to buy a stock at price $K$ (strike) at time $T$ (maturity).

Payoff: $\max(S_T - K, 0)$

In [ ]:
# Option parameters
S0 = 100     # current stock price
K = 105      # strike price
T = 1.0      # maturity (years)
r = 0.05     # risk-free rate
sigma = 0.20 # volatility

# Black-Scholes analytical price
d1 = (np.log(S0 / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
d2 = d1 - sigma * np.sqrt(T)
bs_price = S0 * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)

print(f"Black-Scholes price: ${bs_price:.4f}")

## 2. Classical Monte Carlo

In [ ]:
# Monte Carlo convergence
np.random.seed(42)
sample_sizes = [10, 50, 100, 500, 1000, 5000, 10000, 50000]
mc_prices = []
mc_errors = []

for N in sample_sizes:
    Z = np.random.standard_normal(N)
    ST = S0 * np.exp((r - 0.5 * sigma**2) * T + sigma * np.sqrt(T) * Z)
    payoffs = np.maximum(ST - K, 0) * np.exp(-r * T)
    mc_price = np.mean(payoffs)
    mc_err = np.std(payoffs) / np.sqrt(N)
    mc_prices.append(mc_price)
    mc_errors.append(mc_err)
    print(f"N={N:>6}: price=${mc_price:.4f} ± ${mc_err:.4f}")

print(f"\nTrue price: ${bs_price:.4f}")

In [ ]:
# Plot convergence: error vs sample size
plt.figure(figsize=(8, 5))
plt.loglog(sample_sizes, mc_errors, 'o-', color='#FF5722', label='Classical MC error')

# 1/sqrt(N) reference line
N_ref = np.array(sample_sizes)
plt.loglog(N_ref, mc_errors[0] * np.sqrt(sample_sizes[0]) / np.sqrt(N_ref),
           '--', color='gray', alpha=0.5, label='$1/\sqrt{N}$ reference')

# Hypothetical quantum 1/N convergence
plt.loglog(N_ref, mc_errors[0] * sample_sizes[0] / N_ref,
           '--', color='#009688', alpha=0.7, label='Quantum $1/N$ (projected)')

plt.xlabel('Number of samples / queries')
plt.ylabel('Estimation error ($)')
plt.title('Monte Carlo convergence: classical vs. quantum')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Quantum amplitude estimation (simplified)

The full QAE circuit for option pricing requires:
1. Encoding the log-normal distribution as a quantum state
2. A comparator oracle (mark states where $S_T > K$)
3. QPE on the Grover iterator

Here we demonstrate the core idea with a discretised 3-qubit model
($2^3 = 8$ price bins) and a simplified Grover iterator.

In [ ]:
# Simplified: Grover search for "profitable" outcomes
# Discretise stock price into 8 bins
n_qubits = 3
n_bins = 2**n_qubits

# Define price bins (log-normal discretisation)
price_range = np.linspace(S0 * 0.5, S0 * 1.5, n_bins + 1)
bin_centers = 0.5 * (price_range[:-1] + price_range[1:])

# Which bins are "in the money" (S_T > K)?
in_the_money = bin_centers > K

print("Bin | Price range      | In the money?")
print("-" * 45)
for i, (lo, hi) in enumerate(zip(price_range[:-1], price_range[1:])):
    itm = "✓" if in_the_money[i] else "✗"
    print(f" {i:>2} | ${lo:6.1f} – ${hi:6.1f} | {itm}")

n_itm = sum(in_the_money)
print(f"\n{n_itm}/{n_bins} bins in the money")
print(f"Fraction: {n_itm/n_bins:.3f}")

In [ ]:
# Run a simple Grover circuit to amplify in-the-money states
# This uses the Grover circuit from Cookbook Recipe 06
grover_circuit = """
OPENQASM 2.0;
include "qelib1.inc";

qreg q[3];
creg c[3];

// Superposition
h q[0];
h q[1];
h q[2];

// Oracle: mark states >= 5 (bins 101, 110, 111)
// These are the in-the-money bins in our discretisation
// Phase flip on |101⟩, |110⟩, |111⟩
// Simplified: flip phase when q[2]=1 AND (q[0]=1 OR q[1]=1)
cx q[0], q[1];
ccx q[1], q[2], q[0];
cx q[0], q[1];
cz q[2], q[0];
cz q[2], q[1];

// Diffusion
h q[0]; h q[1]; h q[2];
x q[0]; x q[1]; x q[2];
ccx q[0], q[1], q[2];
cz q[0], q[1];
x q[0]; x q[1]; x q[2];
h q[0]; h q[1]; h q[2];

measure q[0] -> c[0];
measure q[1] -> c[1];
measure q[2] -> c[2];
"""

results = run_qasm(grover_circuit, shots=1024)

print("Measurement results:")
for outcome in sorted(results.keys()):
    bin_idx = int(outcome, 2)
    itm = "← in the money" if bin_idx >= 5 else ""
    print(f"  {outcome} (bin {bin_idx}): {results[outcome]:>4} counts {itm}")

## What's next?

- **Grover deep dive:** the corresponding deep dive chapter
- **QPE deep dive:** the corresponding deep dive chapter
- **Quantum Counting:** the corresponding deep dive chapter — QPE on the Grover iterator
- [Unit 6 — Supply Chains](../manuscript/06-supply-chains.md): QUBO and quantum annealing